In [11]:
import torch
from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM, pipeline
import ipywidgets as widgets
from IPython.display import display, Markdown

# 1) 모델과 토크나이저 로드
model_id = "skt/kogpt2-base-v2"
tokenizer = PreTrainedTokenizerFast.from_pretrained(model_id,
            eos_token='</s>', pad_token='<pad>', unk_token='<unk>')
model = AutoModelForCausalLM.from_pretrained(model_id)

generator = pipeline("text-generation", model=model, tokenizer=tokenizer,
                     device=0 if torch.cuda.is_available() else -1)

# 2) 드롭다운 프롬프트 / UI 위젯 구성
prompts = [
    "오늘은 날씨가 좋아서 ",
    "속보: 국내 주요 IT 기업이 ",
    "옛날 옛적 어느 마을에 ",
    "나는 한국사 선생님입니다. 조선 시대를 ",
    "별빛이 스며든 고요한 바다, 바람이 은밀히 속삭인다 "
]
dropdown = widgets.Dropdown(options=prompts, description="프롬프트:")

temp = widgets.FloatSlider(value=0.8, min=0.1, max=1.5, step=0.1, description="Temperature")
length = widgets.IntSlider(value=60, min=10, max=200, step=10, description="Tokens")
button = widgets.Button(description="생성하기", button_style='info')
output = widgets.Output()

# 3) 문장 생성 로직
def on_click(_):
    output.clear_output(wait=True)
    with output:
        res = generator(dropdown.value,
                        max_new_tokens=int(length.value),
                        do_sample=True,
                        temperature=float(temp.value),
                        repetition_penalty=1.5,
                        no_repeat_ngram_size=3,
                        eos_token_id=tokenizer.eos_token_id)

        result = res[0]["generated_text"].replace('.', '.<br>')
        display(Markdown(f"### 🪄 생성 결과\n\n{result}"))

button.on_click(on_click)

# 4) 화면 표시
display(Markdown("##✍️ KoGPT2 문장 이어쓰기 실습\n\n<br>프롬프트를 선택하고, temperature와 tokens를 조절해보세요.<br><br>"))
display(dropdown, temp, length, button, output)


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


##✍️ KoGPT2 문장 이어쓰기 실습

<br>프롬프트를 선택하고, temperature와 tokens를 조절해보세요.<br><br>

Dropdown(description='프롬프트:', options=('오늘은 날씨가 좋아서 ', '속보: 국내 주요 IT 기업이 ', '옛날 옛적 어느 마을에 ', '나는 한국사 선생님입니다. 조…

FloatSlider(value=0.8, description='Temperature', max=1.5, min=0.1)

IntSlider(value=60, description='Tokens', max=200, min=10, step=10)

Button(button_style='info', description='생성하기', style=ButtonStyle())

Output()